# UC3 Results Viewer

This notebook loads UC3 simulation results from the local results folder and summarizes service rate, wait times, utilization, and cost trade-offs.

Expected results location (relative to this notebook):
- ../results/<scenario>/standard_eval.csv

If no results are found, run:
- python run_uc3_accessibility.py

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
base_dir = Path.cwd()
results_dir = base_dir.parent / "results"
print(f"Notebook dir: {base_dir}")
print(f"Results dir: {results_dir}")
if not results_dir.exists():
    print("Results folder not found. Run: python run_uc3_accessibility.py")
scenario_dirs = sorted([p for p in results_dir.iterdir() if p.is_dir()]) if results_dir.exists() else []
print(f"Scenarios found: {[p.name for p in scenario_dirs]}")

In [ ]:
def _read_eval_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    if "MoD_0" in df.columns:
        return df
    df.columns = [str(c) for c in df.columns]
    return df

def _col(df: pd.DataFrame) -> str:
    return "MoD_0" if "MoD_0" in df.columns else df.columns[0]

def _metric(df: pd.DataFrame, key: str):
    col = _col(df)
    return df.loc[key, col] if key in df.index else None

def _estimate_costs(scenario: str, total_vkm: float) -> float:
    if scenario == "baseline":
        fix_cost = 20 * 20
        var_cost = total_vkm * 0.25
    elif scenario == "mixed_20":
        fix_cost = 16 * 20 + 4 * 35
        var_cost = total_vkm * 0.27
    elif scenario == "mixed_30":
        fix_cost = 14 * 20 + 6 * 35
        var_cost = total_vkm * 0.28
    else:
        fix_cost = 10 * 20 + 10 * 35
        var_cost = total_vkm * 0.30
    return fix_cost + var_cost

In [ ]:
rows = []
missing = []
for scenario_dir in scenario_dirs:
    eval_path = scenario_dir / "standard_eval.csv"
    if not eval_path.exists():
        missing.append(str(eval_path))
        continue
    df = _read_eval_csv(eval_path)
    servicerate = _metric(df, "served online users [%]")
    avg_wait = _metric(df, "waiting time")
    med_wait = _metric(df, "waiting time (median)")
    p90_wait = _metric(df, "waiting time (90% quantile)")
    utilization = _metric(df, "% fleet utilization")
    total_vkm = _metric(df, "total vkm") or 0
    num_users = _metric(df, "number users") or 0
    total_cost = _estimate_costs(scenario_dir.name, total_vkm)
    cost_per_user = total_cost / num_users if num_users else None
    rows.append({
        "Scenario": scenario_dir.name,
        "Service rate [%]": servicerate,
        "Avg wait [min]": avg_wait / 60 if avg_wait is not None else None,
        "Median wait [min]": med_wait / 60 if med_wait is not None else None,
        "P90 wait [min]": p90_wait / 60 if p90_wait is not None else None,
        "Utilization [%]": utilization,
        "Total cost [EUR]": total_cost,
        "Cost per user [EUR]": cost_per_user
    })

if missing:
    print("Missing standard_eval.csv files:")
    for path in missing:
        print(f"- {path}")

results_df = pd.DataFrame(rows)
results_df.sort_values(by="Scenario", inplace=True)
results_df

In [ ]:
if not results_df.empty:
    fig = px.bar(
        results_df,
        x="Scenario",
        y="Service rate [%]",
        title="Service rate by scenario",
        text="Service rate [%]",
        color="Service rate [%]",
        color_continuous_scale="RdYlGn"
    )
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
if not results_df.empty:
    fig2 = px.bar(
        results_df,
        x="Scenario",
        y=["Avg wait [min]", "Median wait [min]", "P90 wait [min]"],
        title="Wait time comparison",
        barmode="group"
    )
    fig2.show()

In [ ]:
if not results_df.empty:
    fig3 = go.Figure()
    fig3.add_trace(go.Bar(
        x=results_df["Scenario"],
        y=results_df["Total cost [EUR]"],
        name="Total cost [EUR]",
        yaxis="y",
        marker_color="lightblue"
    ))
    fig3.add_trace(go.Scatter(
        x=results_df["Scenario"],
        y=results_df["Service rate [%]"],
        name="Service rate [%]",
        yaxis="y2",
        mode="lines+markers",
        marker=dict(size=10, color="red"),
        line=dict(width=3)
    ))
    fig3.update_layout(
        title="Trade-off: cost vs service rate",
        yaxis=dict(title="Total cost [EUR]"),
        yaxis2=dict(title="Service rate [%]", overlaying="y", side="right"),
        legend=dict(x=0.01, y=0.99)
    )
    fig3.show()

Notes:
- Cost parameters are assumptions (20/35 EUR per vehicle-day and 0.25/0.35 EUR per km).
- Update the cost model if you have local cost data.